# Credit Card Fraud Classification

### Fraud Detection Aml Analytics — Banking-AI

This notebook explains each step in plain language so new learners can follow:

## Step 0 — Notebook Header

**Prerequisites:** Basic Python (`pandas`, `numpy`, `matplotlib`), familiarity with `scikit-learn` basics, and basic understanding of fraud detection and anti-money laundering terminology.

**What You Will Learn:**

After completing this notebook, you will be able to:
- Explain the fraud detection and anti-money laundering problem and why the chosen classification approach suits this banking workflow.
- Implement an end-to-end classification pipeline on synthetic data and evaluate performance with domain-appropriate metrics.
- Assess whether the model is operationally viable, considering error costs, fairness, and the limitations of synthetic data.

**Estimated time:** 35–45 minutes

**Collection context:** Fraud detection, AML compliance, anomaly detection, and financial crime analytics in banking.

## Step 1 — Banking Problem Introduction

**Why this notebook matters:** Banks need to detect fraudulent transactions in real-time to protect customers and minimize financial losses.

**Input data used:** Transaction amount, time of day, merchant category, distance from home, and historical spending patterns.

**What we predict:** Binary fraud label (`is_fraud`).

**ML method used:** Random Forest Classifier (excellent for handling imbalanced tabular data).

**Learning goal:** Learn how to handle imbalanced datasets and evaluate models using precision-recall metrics instead of just accuracy.

**Primary stakeholders:** fraud analysts, compliance officers, risk managers, and financial crime investigators.

**Comprehension check:** Before looking at the data, write one sentence describing the core decision this model supports and who benefits from getting it right.

**Domain framing note:** This notebook uses synthetic data for educational purposes. It demonstrates decision-support prototyping, not production-ready deployment.

## Step 2 — Environment Setup

Import libraries and set deterministic seeds for reproducibility.

In [ ]:
# Requires: numpy>=1.24, pandas>=2.0, scikit-learn>=1.3, matplotlib>=3.7, seaborn>=0.13

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve, auc, roc_auc_score, accuracy_score
from sklearn.preprocessing import LabelEncoder

sns.set_theme(style="whitegrid")

RANDOM_SEED = 42
RNG = np.random.default_rng(RANDOM_SEED)
plt.rcParams["figure.figsize"] = (10, 6)

print("Environment ready.")


## Step 3 — Data Creation & Context

Build Synthetic Dataset

Real credit card data is highly sensitive and rarely shared. We generate 10,000 synthetic transactions with realistic features.

In [ ]:
n = 10000
rng = RNG

transaction_id = range(1, n + 1)
amount = rng.gamma(shape=2, scale=50, size=n)  # Most transactions are small
hour = rng.integers(0, 24, n)
category = rng.choice(['Groceries', 'Dining', 'Travel', 'Online Retail', 'Entertainment', 'Services'], n)
dist_from_home = rng.exponential(scale=10, size=n)  # Most transactions are near home
is_international = rng.choice([0, 1], n, p=[0.95, 0.05])
device_type = rng.choice(['Mobile', 'Desktop', 'POS'], n)

# Logic for Fraud (Injection of patterns)
# Fraud is more likely if: 
# 1. Amount is very high
# 2. Distance from home is very high
# 3. Late night (hour 1-4 AM)
# 4. International

risk_score = (
    0.5 * (amount > 500) +
    1.2 * (dist_from_home > 100) +
    0.8 * ((hour >= 1) & (hour <= 4)) +
    1.0 * is_international +
    rng.normal(0, 0.5, n)
)

fraud_prob = 1 / (1 + np.exp(-(risk_score - 2.5)))
is_fraud = (rng.random(n) < fraud_prob).astype(int)

df = pd.DataFrame({
    'amount': amount,
    'hour': hour,
    'category': category,
    'dist_from_home': dist_from_home,
    'is_international': is_international,
    'device_type': device_type,
    'is_fraud': is_fraud
})

print(f"Dataset generated with {n} rows.")
print(f"Fraudulent transactions: {df['is_fraud'].sum()} ({df['is_fraud'].mean()*100:.2f}%)")

## Step 4 — Exploratory Data Analysis

EDA (Exploratory Data Analysis)

Let's look at how fraud relates to transaction amount and distance.

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.boxplot(x='is_fraud', y='amount', data=df)
plt.title('Transaction Amount: Normal vs Fraud')
plt.yscale('log')

plt.subplot(1, 2, 2)
sns.boxplot(x='is_fraud', y='dist_from_home', data=df)
plt.title('Distance from Home: Normal vs Fraud')
plt.yscale('log')

plt.tight_layout()
plt.show()

**Alt-text:** Distribution plot titled "Transaction Amount: Normal vs Fraud" and "Distance from Home: Normal vs Fraud". The chart highlights distributional differences between groups that inform the modelling approach.

## Step 5 — Feature Engineering

Prepare features for modelling. Domain-meaningful transformations help the model learn patterns aligned with banking practice.

In [ ]:
# Encode categorical features
df_encoded = df.copy()
le = LabelEncoder()
df_encoded['category'] = le.fit_transform(df['category'])
df_encoded['device_type'] = le.fit_transform(df['device_type'])

X = df_encoded.drop('is_fraud', axis=1)
y = df_encoded['is_fraud']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y)

# Random Forest is robust to imbalanced data

## Step 6 — Baseline Establishment

A baseline sets the floor that any useful model must beat. Without it, performance numbers have no context.

In [ ]:
# --- Baseline: always predict the majority class ---
baseline_clf = DummyClassifier(strategy='most_frequent', random_state=RANDOM_SEED)
baseline_clf.fit(X_train, y_train)
baseline_pred = baseline_clf.predict(X_test)
baseline_acc = accuracy_score(y_test, baseline_pred)
print(f"Baseline accuracy (majority-class): {baseline_acc:.3f}")
print(f"Any useful model must beat this {baseline_acc:.3f} floor.")

## Step 7 — Model Training & Evaluation

Train the model and compare performance against the baseline.

**Prediction prompt:** Before viewing the results, what performance do you expect and why?

In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED, class_weight='balanced')
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

In [ ]:
print("Classification Report:")
print(classification_report(y_test, y_pred))

precision, recall, _ = precision_recall_curve(y_test, y_proba)
pr_auc = auc(recall, precision)

plt.figure(figsize=(6, 5))
plt.plot(recall, precision, label=f'PR AUC = {pr_auc:.3f}')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend()
plt.tight_layout()
plt.show()
plt.close()

**Alt-text:** Line chart, precision-recall curve titled "Precision-Recall Curve" with Recall on the x-axis and Precision on the y-axis. The curve shows the trade-off between detection rate and false-alarm rate at different thresholds.

## Step 8 — Interpretability & Explainability

Interpret the model in domain terms. A banking professional should be able to assess whether the model's reasoning is credible.

In [ ]:
feat_importances = pd.Series(model.feature_importances_, index=X.columns)
feat_importances.nlargest(10).plot(kind='barh')
plt.title('Top Features Driving Fraud Detection')
plt.tight_layout()
plt.show()
plt.close()

print("\n--- Real-world Scenario ---")
example_normal = X_test[y_test == 0].iloc[0]
example_fraud = X_test[y_test == 1].iloc[0]

prob_normal = model.predict_proba([example_normal])[0][1]
prob_fraud = model.predict_proba([example_fraud])[0][1]

print(f"Normal Transaction Prediction: {prob_normal:.2%} chance of fraud")
print(f"Actual Fraudulent Transaction Prediction: {prob_fraud:.2%} chance of fraud")

**Alt-text:** Bar chart titled "Top Features Driving Fraud Detection". The chart ranks features or categories by magnitude to highlight dominant factors.

Which features matter most for detecting fraud?

## Step 9 — Operational Application

Demonstrate how the model output feeds into a real banking workflow decision.

In [ ]:
# --- Scenario: score representative cases from the test set ---
print("Operational demonstration — model decision support:\n")
proba = model.predict_proba(X_test)[:, 1]
low_idx  = int(np.argmin(proba))
high_idx = int(np.argmax(proba))
mid_idx  = int(np.argsort(proba)[len(proba) // 2])

for label, idx in [("Low-risk", low_idx), ("Medium-risk", mid_idx), ("High-risk", high_idx)]:
    row = X_test.iloc[idx]
    prob = proba[idx]
    print(f"{label} case  predicted probability: {prob:.1%}")
    print(f"  Features: {dict(row.round(2))}")
    print()

print("A decision-maker would combine these scores with business rules and domain judgement.")

## Step 10 — Summary & Reflection

**What you accomplished:** You built an end-to-end fraud detection and anti-money laundering workflow — from problem framing through data creation, baseline comparison, model training, evaluation, and operational demonstration.

**Limitations:**

- The dataset is synthetic and cannot capture the full complexity of real banking data.
- Model performance on synthetic data does not guarantee equivalent performance in production.
- This notebook is an educational prototype. Real deployment requires validated data, regulatory review, and ongoing monitoring.

**Ethical reflection:**

- False positives can freeze legitimate accounts, causing financial hardship and customer distrust.
- Fraud models risk encoding biases against specific demographic groups or geographic regions.
- Transaction monitoring must balance fraud prevention with customer privacy rights.

**Reflection questions:**

1. What additional data sources or features would you need before trusting this model in a live banking environment?
2. How would the relative cost of false positives versus false negatives change the metric you optimise for?
3. How could you adapt this workflow to a related problem in fraud detection and anti-money laundering?

## References

- Scikit-learn documentation: [https://scikit-learn.org/stable/](https://scikit-learn.org/stable/)
- Project quality baseline: `QUALITY_STANDARD.md`
- Domain context drawn from public banking and financial services literature.